# Application Flow and Architectural Overview

This notebook provides a detailed walkthrough of how data and logic flow through the **Practise Application**. It is intended for developers to understand the sequence of operations from the entry point to the database.

## 1. Application Entry Point

The application starts at `PractiseApplication.java`. 

**Key Responsibility:**
- Load environment variables from `.env` using `Dotenv`.
- Map these variables to System properties so Spring Boot can inject them into `application.yml` (e.g., `${DATABASE_URL}`).
- Bootstrap the Spring Context.

## 2. Layered Architecture Flow

The application follows a strict unidirectional flow for every request:

1.  **Client Request**: A JSON payload is sent to a REST endpoint.
2.  **Controller**: Captures the request, performs basic validation (via `@Valid`), and maps it to a **DTO** or **Entity**.
3.  **Service**: Contains the business logic. It coordinates between different repositories and handles transactions.
4.  **Repository**: Communicates with the PostgreSQL database using JPA/Hibernate.
5.  **Database**: Persists or retrieves the data.

## 3. Detailed Example: Placing an Order

The order placement is the most complex flow in the system. Here is the step-by-step logic inside `OrderService.createOrder`:

### Step A: The Request (DTO)
The user sends an `OrderRequest` which contains customer details and a list of `OrderItemRequest` objects (Product ID + Quantity).

### Step B: Logic Execution
1.  **Initialization**: Create a new `Order` entity and set basic info.
2.  **Validation Loop**: For each item in the request:
    - Fetch the `Product` from `ProductRepository`.
    - **Check Stock**: If `stockQuantity < requestedQuantity`, throw a `RuntimeException`.
    - **Price Calculation**: Calculate the sub-total using `BigDecimal` for precision.
    - **Inventory Update**: Subtract the requested quantity from the product's stock.
3.  **Entity Mapping**: Create `OrderItem` entities linked to both the `Order` and the `Product`.
4.  **Finalization**: Set the total price and the list of items on the `Order` object.

### Step C: Transactional Save
The method is marked `@Transactional`. If any step fails (e.g., the last item in a 10-item list is out of stock), the database rolls back everything. No partial orders are created, and stock is not deducted incorrectly.

## 4. API Endpoints Reference

### Product API (`/api/products`)

| Method | Endpoint | Description | Payload |
| :--- | :--- | :--- | :--- |
| POST | `/` | Create a new product | `Product` (Entity) |
| GET | `/` | List all products | N/A |
| GET | `/{id}` | Get specific product | N/A |
| PUT | `/{id}` | Update product details | `Product` (Entity) |
| DELETE | `/{id}` | Remove a product | N/A |

### Order API (`/api/orders`)

| Method | Endpoint | Description | Payload |
| :--- | :--- | :--- | :--- |
| POST | `/` | Place a new order | `OrderRequest` (DTO) |

## 5. Summary of Key Services

### `ProductService` 
- **Purpose**: Direct CRUD management.
- **Key Logic**: Handles the `ProductNotFound` exceptions and maps updates from the request to existing database records.

### `OrderService` 
- **Purpose**: Complex business orchestration.
- **Key Logic**: 
    - Precision math with `BigDecimal`.
    - Cross-entity coordination (linking `Order` with `Product` via `OrderItem`).
    - State management (Stock deduction).

## 6. Data Integrity Notes

- **DTO vs Entity**: `OrderRequest` is used for incoming data to prevent users from manually setting fields like `id` or `totalPrice`.
- **Circular References**: `@JsonManagedReference` and `@JsonBackReference` are used in the entities to prevent infinite loops when Jackson serializes the Order <-> OrderItem relationship to JSON.